# NSU BDA. 2024 Accidents
Соревнование для студенттов курса АБМД, ФИТ НГУ 2024

Студент: Митюшин Владимир 24221

In [127]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Input

Загрузим данные

In [128]:
X_train = pd.read_csv('../data/X_train.csv', index_col=0)
X_testFinal = pd.read_csv('../data/X_test.csv', index_col=0)
Y_train = pd.read_csv('../data/Y_train.csv', index_col=0)

Выведем информацию по датасетам

In [129]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 499056 entries, 241270 to 121958
Data columns (total 35 columns):
 #   Column                                   Non-Null Count   Dtype 
---  ------                                   --------------   ----- 
 0   accident_index                           499056 non-null  object
 1   vehicle_reference                        499056 non-null  int64 
 2   casualty_class                           499056 non-null  int64 
 3   sex_of_casualty                          499056 non-null  int64 
 4   age_of_casualty                          499056 non-null  int64 
 5   age_band_of_casualty                     499056 non-null  int64 
 6   pedestrian_location                      499056 non-null  int64 
 7   pedestrian_movement                      499056 non-null  int64 
 8   car_passenger                            499056 non-null  int64 
 9   bus_or_coach_passenger                   499056 non-null  int64 
 10  pedestrian_road_maintenance_worker       499

In [130]:
Y_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 499056 entries, 241270 to 121958
Data columns (total 1 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   casualty_severity  499056 non-null  int64
dtypes: int64(1)
memory usage: 7.6 MB


Посмотрим более подробно на целевой признак

In [131]:
print("min: ", Y_train['casualty_severity'].min())
print("max: ", Y_train['casualty_severity'].max())
print("unique: ", Y_train['casualty_severity'].unique())

min:  1
max:  3
unique:  [3 2 1]


### Исходя из возможных значений целевого признака понимаем, что имеем задачу классификации

## Очистка и подготовка данных

In [132]:
X_train.describe()

,vehicle_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,pedestrian_location,pedestrian_movement,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,...,speed_limit,junction_detail,junction_control,second_road_class,second_road_number,pedestrian_crossing_human_control,pedestrian_crossing_physical_facilities,light_conditions,weather_conditions,road_surface_conditions
count,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,...,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000
mean,1.457009,1.472198,1.366891,36.823471,6.312548,0.758696,0.613450,0.231964,0.050407,0.025949,...,37.549017,3.777957,1.678210,2.994840,223.466781,0.321896,1.104309,2.056154,1.638289,1.371584
std,2.323518,0.725208,0.529874,19.526246,2.453729,2.144590,1.963437,0.627427,0.436082,0.232426,...,14.742382,11.998430,2.516433,2.758235,935.872042,1.637604,2.372924,1.734888,1.790234,0.930508
min,1.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
25%,1.000000,1.000000,1.000000,22.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,30.000000,0.000000,-1.000000,0.000000,-1.000000,0.000000,0.000000,1.000000,1.000000,1.000000
50%,1.000000,1.000000,1.000000,34.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,30.000000,1.000000,2.000000,3.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
75%,2.000000,2.000000,2.000000,50.000000,8.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,50.000000,3.000000,4.000000,6.000000,0.000000,0.000000,0.000000,4.000000,1.000000,2.000000
max,999.000000,3.000000,9.000000,102.000000,11.000000,10.000000,9.000000,9.000000,9.000000,2.000000,...,70.000000,99.000000,9.000000,6.000000,9999.000000,9.000000,9.000000,7.000000,9.000000,9.000000


In [133]:
df = X_train.copy()

In [134]:
# df.hist(figsize=(20, 15));

In [135]:
# df[df.select_dtypes('number').columns].corr().style.background_gradient(cmap='coolwarm')

In [136]:
drop_indexes = df.index[df['vehicle_reference'] > 20].tolist()
drop_indexes= drop_indexes + df.index[df['age_of_vehicle'] > 25].tolist()
df.drop(drop_indexes, inplace=True)
Y_train.drop(drop_indexes, inplace=True)

df['sex_of_casualty'] = df['sex_of_casualty'].replace(9, -1)
df['car_passenger'] = df['car_passenger'].replace(9, -1)
df['casualty_type'] = df['casualty_type'].replace([3, 4, 5, 22, 23, 97, 103, 104, 105, 106], 2).replace([20, 21, 98, 113], 19)
df.describe()

,vehicle_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,pedestrian_location,pedestrian_movement,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,...,speed_limit,junction_detail,junction_control,second_road_class,second_road_number,pedestrian_crossing_human_control,pedestrian_crossing_physical_facilities,light_conditions,weather_conditions,road_surface_conditions
count,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,...,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.00000,496651.000000
mean,1.449964,1.473234,1.367147,36.783472,6.307860,0.760866,0.615100,0.220797,0.050619,0.026020,...,37.519067,3.786178,1.679829,2.996621,223.642938,0.323237,1.107128,2.057342,1.63932,1.372052
std,0.593538,0.725776,0.527032,19.515047,2.453348,2.147334,1.965768,0.553609,0.436992,0.232706,...,14.734020,12.024648,2.516757,2.757975,936.358471,1.640897,2.375478,1.735276,1.79187,0.931780
min,1.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.00000,-1.000000
25%,1.000000,1.000000,1.000000,22.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,30.000000,0.000000,-1.000000,0.000000,-1.000000,0.000000,0.000000,1.000000,1.00000,1.000000
50%,1.000000,1.000000,1.000000,34.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,30.000000,1.000000,2.000000,3.000000,0.000000,0.000000,0.000000,1.000000,1.00000,1.000000
75%,2.000000,2.000000,2.000000,50.000000,8.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,50.000000,3.000000,4.000000,6.000000,0.000000,0.000000,0.000000,4.000000,1.00000,2.000000
max,20.000000,3.000000,2.000000,102.000000,11.000000,10.000000,9.000000,2.000000,9.000000,2.000000,...,70.000000,99.000000,9.000000,6.000000,9999.000000,9.000000,9.000000,7.000000,9.00000,9.000000


In [137]:
df.drop(columns='accident_index', inplace=True)

In [138]:
df = df.drop(columns=['age_of_casualty', 'pedestrian_movement', 'second_road_number', 'junction_control', 'pedestrian_crossing_human_control', 'engine_capacity_cc'])

In [139]:
# df = df.drop(columns=['bus_or_coach_passenger', 'pedestrian_road_maintenance_worker', 'vehicle_left_hand_drive', 'vehicle_reference', 'junction_detail']) # , 'vehicle_type'

In [140]:
df.describe()

,vehicle_reference,casualty_class,sex_of_casualty,age_band_of_casualty,pedestrian_location,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_type,casualty_home_area_type,...,first_road_class,first_road_number,road_type,speed_limit,junction_detail,second_road_class,pedestrian_crossing_physical_facilities,light_conditions,weather_conditions,road_surface_conditions
count,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,...,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.00000,496651.000000
mean,1.449964,1.473234,1.367147,6.307860,0.760866,0.220797,0.050619,0.026020,7.069497,1.066266,...,4.144252,800.691008,5.232447,37.519067,3.786178,2.996621,1.107128,2.057342,1.63932,1.372052
std,0.593538,0.725776,0.527032,2.453348,2.147334,0.553609,0.436992,0.232706,9.218434,0.918710,...,1.470665,1593.561288,1.673314,14.734020,12.024648,2.757975,2.375478,1.735276,1.79187,0.931780
min,1.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,1.000000,0.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.00000,-1.000000
25%,1.000000,1.000000,1.000000,5.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,...,3.000000,0.000000,6.000000,30.000000,0.000000,0.000000,0.000000,1.000000,1.00000,1.000000
50%,1.000000,1.000000,1.000000,6.000000,0.000000,0.000000,0.000000,0.000000,9.000000,1.000000,...,4.000000,38.000000,6.000000,30.000000,1.000000,3.000000,0.000000,1.000000,1.00000,1.000000
75%,2.000000,2.000000,2.000000,8.000000,0.000000,0.000000,0.000000,0.000000,9.000000,1.000000,...,6.000000,562.000000,6.000000,50.000000,3.000000,6.000000,0.000000,4.000000,1.00000,2.000000
max,20.000000,3.000000,2.000000,11.000000,10.000000,2.000000,9.000000,2.000000,99.000000,3.000000,...,6.000000,9176.000000,9.000000,70.000000,99.000000,6.000000,9.000000,7.000000,9.00000,9.000000


In [141]:
# import holoviews as hv
# from holoviews import dim
# from holoviews import opts
# hv.extension('bokeh')

# def f(x):
#     return hv.BoxWhisker(df[x]).opts(height=120, responsive=True, toolbar='above', invert_axes=True, tools=['hover'])

# hv.DynamicMap(f, kdims=['x']).redim.values(x=df.select_dtypes('number').columns)

In [142]:
# df.hist(figsize=(20, 11));

### Посмотрим на категориальные признаки

In [143]:
# df.describe(include='O')

In [144]:
df['generic_make_model'] = df['generic_make_model'].str.split(n=1).str[0].str.upper()
display(df['generic_make_model'].value_counts(dropna=False))

generic_make_model
-1            131363
FORD           43959
VAUXHALL       36307
VOLKSWAGEN     28299
HONDA          25694
               ...  
CASE               7
OPEL               6
PROTON             6
VALTRA             5
VESPA              4
Name: count, Length: 114, dtype: int64

In [145]:
# display(df['local_authority_highway'].value_counts(dropna=False))

In [146]:
df['local_authority_highway'] = df['local_authority_highway'].str[:4]
display(df['local_authority_highway'].value_counts(dropna=False))

local_authority_highway
E100    158518
E060    115077
E090    100761
E080     83493
S120     21799
W060     16905
EHEA        98
Name: count, dtype: int64

In [147]:
# df.drop(columns=['local_authority_highway', 'generic_make_model'], inplace=True)

In [148]:
one_hot = pd.get_dummies(df.select_dtypes('O'), prefix=df.select_dtypes('O').columns, dtype=bool)
df = pd.concat([one_hot, df.select_dtypes('number'), df.select_dtypes('bool')], axis=1)

df.describe()

,vehicle_reference,casualty_class,sex_of_casualty,age_band_of_casualty,pedestrian_location,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_type,casualty_home_area_type,...,first_road_class,first_road_number,road_type,speed_limit,junction_detail,second_road_class,pedestrian_crossing_physical_facilities,light_conditions,weather_conditions,road_surface_conditions
count,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,...,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.00000,496651.000000
mean,1.449964,1.473234,1.367147,6.307860,0.760866,0.220797,0.050619,0.026020,7.069497,1.066266,...,4.144252,800.691008,5.232447,37.519067,3.786178,2.996621,1.107128,2.057342,1.63932,1.372052
std,0.593538,0.725776,0.527032,2.453348,2.147334,0.553609,0.436992,0.232706,9.218434,0.918710,...,1.470665,1593.561288,1.673314,14.734020,12.024648,2.757975,2.375478,1.735276,1.79187,0.931780
min,1.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,1.000000,0.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.00000,-1.000000
25%,1.000000,1.000000,1.000000,5.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,...,3.000000,0.000000,6.000000,30.000000,0.000000,0.000000,0.000000,1.000000,1.00000,1.000000
50%,1.000000,1.000000,1.000000,6.000000,0.000000,0.000000,0.000000,0.000000,9.000000,1.000000,...,4.000000,38.000000,6.000000,30.000000,1.000000,3.000000,0.000000,1.000000,1.00000,1.000000
75%,2.000000,2.000000,2.000000,8.000000,0.000000,0.000000,0.000000,0.000000,9.000000,1.000000,...,6.000000,562.000000,6.000000,50.000000,3.000000,6.000000,0.000000,4.000000,1.00000,2.000000
max,20.000000,3.000000,2.000000,11.000000,10.000000,2.000000,9.000000,2.000000,99.000000,3.000000,...,6.000000,9176.000000,9.000000,70.000000,99.000000,6.000000,9.000000,7.000000,9.00000,9.000000


In [149]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 496651 entries, 241270 to 121958
Columns: 147 entries, generic_make_model_-1 to road_surface_conditions
dtypes: bool(121), int64(26)
memory usage: 159.6 MB


## Перейдём к обучению

In [150]:
df_all = df.copy()
df_all['y'] = Y_train['casualty_severity']
df_2 = df_all[df_all['y'] != 1]

df_all = df_all.drop(columns='y')
df_2 = df_2.drop(columns='y')

Y_all = Y_train.copy()
Y_2 = Y_all.drop(index=Y_all.index[Y_all['casualty_severity'] == 1])
Y_1 = Y_all['casualty_severity'].replace(3, 2)
Y_2 = Y_2.apply(lambda y: y-2)
Y_1 = Y_1.apply(lambda y: y-1)

print(df_2.shape, Y_2.shape)
print(df_all.shape, Y_1.shape)
print(np.unique(Y_1), np.unique(Y_2))

(490725, 147) (490725, 1)
(496651, 147) (496651,)
[0 1] [0 1]


In [151]:
X_train, X_test, y_train, y_test = train_test_split(df_2, Y_2, random_state=42) # Y_train
print(X_train.shape, y_train.shape)

y_train = keras.utils.to_categorical(y_train, num_classes=2) # _resampled
y_test = keras.utils.to_categorical(y_test, num_classes=2)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

(368043, 147) (368043, 1)
y_train shape: (368043, 2)
y_test shape: (122682, 2)


In [152]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train) # _resampled
X_test = scaler.transform(X_test)

In [153]:
print(X_train.shape, X_testFinal.shape)

(368043, 147) (166352, 35)


In [162]:
def create_model(seed):
    keras.utils.set_random_seed(seed)
    model = Sequential() 
    model.add(Input(shape=(147, 1)))
    model.add(Flatten())
    
    model.add(Dense(500, activation='elu')) # 'relu', 'leaky_relu', 'elu'
    model.add(Dense(220, activation='elu')) # 'relu', 'leaky_relu', 'elu'
    model.add(Dropout(.5))
    model.add(Dense(140, activation='leaky_relu')) # 'relu', 'leaky_relu', 'elu'
    model.add(Dense(90, activation='leaky_relu'))
    model.add(Dropout(.25))
    model.add(Dense(32, activation='leaky_relu'))
    model.add(Dense(16, activation='leaky_relu'))
    model.add(Dense(2, activation='softmax')) # 'sigmoid', 'softmax', 'tanh'
    model.summary()
    
    model.compile(optimizer=keras.optimizers.Adam(), 
                  loss='categorical_crossentropy', metrics=['f1_score'])
    return model

In [163]:
model = create_model(142)

class_weights = {
    0: 0.9,
    1: 0.4
}

history = model.fit(X_train, y_train,
                    epochs=15,
                    validation_data=(X_test, y_test),
                    batch_size=1000,
                    class_weight=class_weights,
                    verbose=2
                    )

Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_13 (Flatten)            │ (None, 147)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_80 (Dense)                │ (None, 500)            │        74,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_81 (Dense)                │ (None, 220)            │       110,220 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_26 (Dropout)            │ (None, 220)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_82 (Dense)                │ (None, 140)            │        30,940 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_83 (Dense)                │ (None, 90)             │        12,690 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_27 (Dropout)            │ (None, 90)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_84 (Dense)                │ (None, 32)             │         2,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_85 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_86 (Dense)                │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 231,324 (903.61 KB)

 Trainable params: 231,324 (903.61 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
369/369 - 6s - 17ms/step - f1_score: 0.5878 - loss: 0.2877 - val_f1_score: 0.6140 - val_loss: 0.4951
Epoch 2/15
369/369 - 4s - 12ms/step - f1_score: 0.6060 - loss: 0.2813 - val_f1_score: 0.6131 - val_loss: 0.4898
Epoch 3/15
369/369 - 4s - 11ms/step - f1_score: 0.6082 - loss: 0.2798 - val_f1_score: 0.6130 - val_loss: 0.4844
Epoch 4/15
369/369 - 4s - 11ms/step - f1_score: 0.6100 - loss: 0.2788 - val_f1_score: 0.6147 - val_loss: 0.4819
Epoch 5/15
369/369 - 4s - 11ms/step - f1_score: 0.6121 - loss: 0.2780 - val_f1_score: 0.6194 - val_loss: 0.4870
Epoch 6/15
369/369 - 4s - 11ms/step - f1_score: 0.6129 - loss: 0.2774 - val_f1_score: 0.6187 - val_loss: 0.4850
Epoch 7/15
369/369 - 4s - 12ms/step - f1_score: 0.6143 - loss: 0.2771 - val_f1_score: 0.6196 - val_loss: 0.4868
Epoch 8/15
369/369 - 4s - 12ms/step - f1_score: 0.6155 - loss: 0.2767 - val_f1_score: 0.6163 - val_loss: 0.4803
Epoch 9/15
369/369 - 4s - 11ms/step - f1_score: 0.6157 - loss: 0.2764 - val_f1_score: 0.6208 - val_loss:

In [156]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(df_all, Y_1, random_state=42)
X_train1 = scaler.fit_transform(X_train1)
X_test1 = scaler.transform(X_test1)

y_train1 = keras.utils.to_categorical(y_train1, num_classes=2)
y_test1 = keras.utils.to_categorical(y_test1, num_classes=2)
print('y_train shape:', y_train1.shape)
print('y_test shape:', y_test1.shape)
print(X_train1.shape, y_train1.shape)

y_train shape: (372488, 2)
y_test shape: (124163, 2)
(372488, 147) (372488, 2)


In [164]:
model2 = create_model(242)

class_weights1 = {
    0: 8.5,
    1: 0.8
}

history2 = model2.fit(X_train1, y_train1,
                    epochs=15,
                    validation_data=(X_test1, y_test1),
                    batch_size=1000,
                    class_weight=class_weights1,
                    verbose=2
                    )

Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_14 (Flatten)            │ (None, 147)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_87 (Dense)                │ (None, 500)            │        74,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_88 (Dense)                │ (None, 220)            │       110,220 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_28 (Dropout)            │ (None, 220)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_89 (Dense)                │ (None, 140)            │        30,940 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_90 (Dense)                │ (None, 90)             │        12,690 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_29 (Dropout)            │ (None, 90)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_91 (Dense)                │ (None, 32)             │         2,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_92 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_93 (Dense)                │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 231,324 (903.61 KB)

 Trainable params: 231,324 (903.61 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
373/373 - 6s - 16ms/step - f1_score: 0.5350 - loss: 0.2714 - val_f1_score: 0.5559 - val_loss: 0.1642
Epoch 2/15
373/373 - 4s - 12ms/step - f1_score: 0.5516 - loss: 0.2531 - val_f1_score: 0.5523 - val_loss: 0.1526
Epoch 3/15
373/373 - 4s - 12ms/step - f1_score: 0.5606 - loss: 0.2477 - val_f1_score: 0.5597 - val_loss: 0.1453
Epoch 4/15
373/373 - 4s - 12ms/step - f1_score: 0.5622 - loss: 0.2444 - val_f1_score: 0.5585 - val_loss: 0.1367
Epoch 5/15
373/373 - 4s - 12ms/step - f1_score: 0.5630 - loss: 0.2428 - val_f1_score: 0.5609 - val_loss: 0.1406
Epoch 6/15
373/373 - 4s - 11ms/step - f1_score: 0.5670 - loss: 0.2411 - val_f1_score: 0.5569 - val_loss: 0.1344
Epoch 7/15
373/373 - 4s - 11ms/step - f1_score: 0.5663 - loss: 0.2400 - val_f1_score: 0.5597 - val_loss: 0.1336
Epoch 8/15
373/373 - 4s - 11ms/step - f1_score: 0.5669 - loss: 0.2386 - val_f1_score: 0.5654 - val_loss: 0.1307
Epoch 9/15
373/373 - 4s - 11ms/step - f1_score: 0.5719 - loss: 0.2373 - val_f1_score: 0.5633 - val_loss:

In [158]:
X_testFinal = pd.read_csv('../data/X_test.csv', index_col=0)

X_testFinal = X_testFinal.reindex(df_all.columns, axis=1, fill_value=0)
X_testFinal_data = scaler.transform(X_testFinal)
X_testFinal.describe()

,generic_make_model_-1,generic_make_model_ABARTH,generic_make_model_AJS,generic_make_model_ALEXANDER,generic_make_model_ALFA,generic_make_model_APRILIA,generic_make_model_AUDI,generic_make_model_BENELLI,generic_make_model_BENTLEY,generic_make_model_BMW,...,first_road_class,first_road_number,road_type,speed_limit,junction_detail,second_road_class,pedestrian_crossing_physical_facilities,light_conditions,weather_conditions,road_surface_conditions
count,166352.0,166352.0,166352.0,166352.0,166352.0,166352.0,166352.0,166352.0,166352.0,166352.0,...,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000
mean,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4.149598,796.766651,5.238975,37.547730,3.767667,3.008446,1.095220,2.051848,1.634257,1.371369
std,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.469986,1585.718521,1.665436,14.710353,11.917428,2.759564,2.362987,1.732798,1.787258,0.933177
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.000000,0.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
25%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3.000000,0.000000,6.000000,30.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
50%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4.000000,38.000000,6.000000,30.000000,2.000000,3.000000,0.000000,1.000000,1.000000,1.000000
75%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6.000000,567.000000,6.000000,50.000000,3.000000,6.000000,0.000000,4.000000,1.000000,2.000000
max,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6.000000,9480.000000,9.000000,70.000000,99.000000,6.000000,9.000000,7.000000,9.000000,9.000000


In [165]:
import random

y1_final = model.predict(X_testFinal_data)
y2_final = model2.predict(X_testFinal_data)
tmp1 = []
tmp2 = []
for i in range(len(y1_final)):
    tmp1.append(np.argmax(y1_final[i])+2)
    tmp2.append(np.argmax(y2_final[i])+1)
print(np.unique(tmp1), np.unique(tmp2))

y_final = [random.randint(1, 3) for i in range(X_testFinal_data.shape[0])]
for i in range(X_testFinal_data.shape[0]):
    if tmp2[i] == 1:
        y_final[i] = 1
    else: 
        y_final[i] = tmp1[i]

print(np.unique(y_final), y_final)

5199/5199 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step
5199/5199 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step
[2 3] [1 2]
[1 2 3] [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 3, 3, 2, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 1, 3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 2, 3, 3, 2, 3, 3, 1, 3, 2, 2, 3, 3, 3, 3, 3, 3, 3, 2, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 2, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 2, 3, 2, 3, 2, 3, 2, 3, 3, 2, 2, 3, 2, 3, 3, 3, 3, 3, 3, 2, 3, 3, 2, 2, 3, 2, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 2, 3, 3, 2, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 2, 3, 3, 3,

In [166]:
my_file = open("../predicts/nn_balance6.csv", "w+")

my_file.write("Id,casualty_severity\n")
my_i = 0
for i, row in X_testFinal.iterrows():
    my_file.write(f"{i}, {y_final[my_i]}\n")
    my_i = my_i+1
my_file.close()